# Unsafe Lifting Posture Detection Using CNN and Pose Estimation

**Google Colab Training, Threshold Optimization, and Evaluation Pipeline**

This notebook provides the official Google Colab workflow to train the 2D CNN posture classifier on genuine manual lifting videos. It enforces:
1. MediaPipe PoseLandmarker body-keypoint extraction
2. Standardized 128x128 single-channel skeletal pose-map generation
3. **Participant-grouped splitting** to strictly prevent data leakage across subjects
4. 2D CNN architecture training with EarlyStopping and best checkpoint saving
5. **Validation-only decision threshold calibration**
6. Genuine held-out test evaluation (Accuracy, Precision, Recall, F1, Specificity, ROC-AUC, Confusion Matrix)
7. Export of model artifacts for local Streamlit deployment

## 1. Environment Setup & Dependency Installation

In [ ]:
!pip install --quiet mediapipe opencv-python-headless tensorflow scikit-learn matplotlib seaborn pandas

## 2. Mount Google Drive or Upload Dataset

In [ ]:
import os
from pathlib import Path

# Optional: Mount Google Drive if your dataset is hosted there
# from google.colab import drive
# drive.mount('/content/drive')
# WORKSPACE = Path('/content/drive/MyDrive/unsafe-lifting-detection')

WORKSPACE = Path('/content/unsafe-lifting-detection')
WORKSPACE.mkdir(parents=True, exist_ok=True)
os.chdir(WORKSPACE)
print("Working directory:", os.getcwd())

## 3. Download MediaPipe Pose Landmarker Model Asset

In [ ]:
import urllib.request

models_dir = Path("models")
models_dir.mkdir(parents=True, exist_ok=True)
task_path = models_dir / "pose_landmarker_lite.task"

if not task_path.exists():
    url = "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/latest/pose_landmarker_lite.task"
    print("Downloading MediaPipe model asset...")
    urllib.request.urlretrieve(url, str(task_path))
    print(f"Downloaded: {task_path.stat().st_size} bytes")
else:
    print("MediaPipe asset already present.")

## 4. Validate Dataset Annotations & Verify Integrity

In [ ]:
import sys
if str(WORKSPACE) not in sys.path:
    sys.path.insert(0, str(WORKSPACE))

from src.dataset_loader import validate_annotations

ann_file = Path("data/annotations.csv")
if ann_file.exists():
    report = validate_annotations(ann_file)
    print("Validation Status:", "VALID" if report.is_valid else "INVALID")
    print(report.to_dict())
else:
    print("data/annotations.csv not found. Please upload your video dataset and annotations.")

## 5. Extract Pose Maps & Partition Without Data Leakage

In [ ]:
from src.dataset_loader import build_and_cache_dataset

df_meta, report = build_and_cache_dataset(
    annotations_csv=ann_file,
    sample_rate_fps=5
)
print(f"Extracted {len(df_meta)} valid pose frames across subjects.")

## 6. Train 2D CNN & Optimize Decision Threshold

In [ ]:
import pandas as pd
from src.train import train_model

train_df = pd.read_csv("data/splits/train.csv")
val_df = pd.read_csv("data/splits/val.csv")

model, summary = train_model(
    train_df=train_df,
    val_df=val_df,
    epochs=35,
    batch_size=16,
    learning_rate=1e-3,
    seed=42
)

print("Training Complete!")
print("Optimal Validation Threshold:", summary["decision_threshold"])

## 7. Evaluate Frozen Model on Held-Out Test Subjects

In [ ]:
from src.evaluate import evaluate_held_out_test_set

test_metrics = evaluate_held_out_test_set()
print("Held-Out Test Set Evaluation Results:")
for k, v in test_metrics.items():
    print(f"  {k}: {v}")

## 8. Export Model Artifacts for Streamlit Dashboard

In [ ]:
import shutil
from google.colab import files

export_zip = WORKSPACE / "lifting_model_artifacts.zip"
archive_paths = ["models", "outputs"]

# Zip models/ and outputs/ directories
shutil.make_archive(str(WORKSPACE / "lifting_model_artifacts"), 'zip', root_dir=str(WORKSPACE))
print("Created artifact archive:", export_zip)
files.download(str(WORKSPACE / "lifting_model_artifacts.zip"))